# Практика: ALE

В уроке «ALE» мы разобрали, почему PDP ошибается на скоррелированных признаках, как ALE исправляет это разностями прогнозов внутри интервалов, и посчитали ALE руками на 19 покупательницах. В этой тетради ALE считается сначала на числах из урока, затем на настоящих данных, проверяется на линейной модели и сравнивается с PDP и M-plot на синтетическом примере, где правильный ответ известен заранее.

В каждом задании есть ячейка с `...` — ее нужно дописать. Задания тренажера по этой тетради — в уроке «ALE: Практика» модуля «Методы объяснений для моделей машинного обучения pt. 1» на платформе курса.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence

%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## 1. ALE на числах из урока

В шаге «ALE с 0» урока «ALE» возраст 19 покупательниц разбит на три интервала по 6, 7 и 6 наблюдений. Для интервала 2 модель дала прогнозы трат на нижней границе (возраст 30) и на верхней (возраст 37) — они в ячейке ниже. Локальные эффекты интервалов 1 и 3 из урока: $\Delta_1 = 19.33$, $\Delta_3 = 7.67$.

**Задание 1.** Посчитайте $\Delta_2$ (округлите до сотых, как в уроке), накопленный ALE $A_k = \sum_{l \le k} \Delta_l$ и центрированный ALE

$$A_k - \bar A,\qquad \bar A = \frac{1}{n}\sum_{k} |N(k)|\, A_k.$$

In [ ]:
pred_lower = np.array([61, 40, 40, 88, 70, 39, 61])  # возраст = 30
pred_upper = np.array([34, 76, 44, 76, 51, 98, 58])  # возраст = 37
counts = np.array([6, 7, 6])

delta_2 = ...  # ваш код: средняя разность прогнозов на границах интервала 2, до сотых
delta = np.array([19.33, delta_2, 7.67])
A = ...      # ваш код: накопленный ALE
A_bar = ...  # ваш код: среднее A, взвешенное на число объектов в интервалах
print("Delta_2 =", delta_2)
print("A =", A.round(2), " A_bar =", round(A_bar, 2))
print("центрированный ALE =", (A - A_bar).round(2))

## 2. ALE на настоящих данных



### Данные и модель

Набор California Housing встроен в `sklearn`: 20 640 районов Калифорнии из переписи 1990 года, восемь признаков района, целевая переменная — медианная стоимость дома в районе в сотнях тысяч долларов. Признаки: `MedInc` — медианный доход (десятки тысяч долларов), `HouseAge` — медианный возраст дома, `AveRooms` и `AveBedrms` — среднее число комнат и спален на дом, `Population` — население района, `AveOccup` — среднее число жильцов на дом, `Latitude` и `Longitude` — широта и долгота.

Делим данные на обучающую и тестовую части (80% и 20%, `random_state=42`) и обучаем градиентный бустинг `HistGradientBoostingRegressor` с параметрами по умолчанию. Для методов этой тетради модель — черный ящик: от нее нужен только прогноз `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"R^2 модели на тестовых данных: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

Диапазон признака делим на $K$ интервалов по квантилям, чтобы в каждый попало примерно поровну объектов. Соседние интервалы делят общую границу $z_k$. Для объекта из интервала $k$ считаем разность прогнозов на правой и левой границах, остальные признаки объекта не трогаем; среднее по объектам интервала — локальный эффект $\Delta_k$. Кривую накапливаем от левого края, $A_0 = 0$, и центрируем так, чтобы ее среднее по объектам было нулем (значение объекта в интервале — середина отрезка между $A_{k-1}$ и $A_k$). Так центрируют пакеты `PyALE` и `ALEPlot`; в статье Apley и Zhu (формула 16) вычитают $\frac1n\sum_k |N(k)|\,A_k$, и кривые отличаются только сдвигом на константу.

**Задание 2.** Допишите функцию `ale`. Она возвращает границы интервалов `z` и центрированные значения ALE в этих границах. Проверьте счетчиком, что ALE стоит $2n$ прогнозов при любом $K$.

In [ ]:
def ale(model, X, feature, K=20):
    x = X[feature].values
    z = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)))
    k = np.clip(np.searchsorted(z, x, side="left") - 1, 0, len(z) - 2)  # номер интервала объекта
    lo, hi = X.copy(), X.copy()
    lo[feature], hi[feature] = z[k], z[k + 1]
    n_k = np.bincount(k, minlength=len(z) - 1)
    diff = ...   # ваш код: разность прогнозов на правой и левой границах
    delta = ...  # ваш код: среднее diff по объектам каждого интервала
    A = ...      # ваш код: накопленная кривая длины len(z), A[0] = 0
    mid = (A[:-1] + A[1:]) / 2
    return z, A - (mid * n_k).sum() / n_k.sum()


class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        self.rows += len(X)
        return self.model.predict(X)


for K in [10, 50]:
    counted = CountingModel(model)
    ale(counted, X_test, "MedInc", K=K)
    print(f"K = {K}: прогнозов {counted.rows}, 2n = {2 * len(X_test)}")

## 3. ALE линейной модели

В уроке «ALE» показано: у линейной модели разность прогнозов на границах интервала равна $w_j (z_k - z_{k-1})$ у всех объектов, и ALE — прямая с наклоном $w_j$ при любой связи признаков.

**Задание 3.** Обучите `LinearRegression`, посчитайте ALE для всех восьми признаков, оцените наклон каждой кривой (`np.polyfit` степени 1) и сравните с весами `lin.coef_`.

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
rows = {}
for j, feature in enumerate(X.columns):
    z, a = ale(lin, X_test, feature)
    slope = ...  # ваш код: наклон прямой, подогнанной к ALE
    rows[feature] = {"наклон ALE": slope, "вес модели": lin.coef_[j]}
pd.DataFrame(rows).T

In [ ]:
z, a = ale(lin, X_test, "MedInc")
w = lin.coef_[0]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(z, a, color=BLUE, lw=2, marker="o", ms=4, label="ALE")
ax.plot(z, w * (z - z.mean()) + a.mean(), color=ORANGE, lw=1.2, ls="--", label=f"прямая с наклоном w = {w:.3f}")
for q in np.quantile(X_test["MedInc"], np.linspace(0.1, 0.9, 9)):
    ax.axvline(q, ymin=0, ymax=0.04, color=MUTED, lw=1)
ax.set_xlabel("MedInc — медианный доход района, десятки тысяч долларов")
ax.set_ylabel("ALE, сотни тысяч долларов")
ax.set_title("ALE по MedInc для линейной регрессии на California Housing")
ax.legend()
plt.show()

## 4. ALE бустинга

**Задание 4.** Постройте ALE градиентного бустинга для признаков `MedInc`, `AveOccup`, `HouseAge`, `Latitude` на тестовых данных, $K = 20$. Внизу каждого графика отметьте децили признака, ось $x$ ограничьте 1-м и 99-м процентилями: у `AveOccup` последний интервал тянется до 1243, и без этого остальные интервалы сливаются у левого края. Посчитайте размах каждой кривой (максимум минус минимум).

In [ ]:
features = ["MedInc", "AveOccup", "HouseAge", "Latitude"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
span = {}
for ax, feature in zip(axes.ravel(), features):
    z, a = ...  # ваш код: ALE бустинга по признаку
    span[feature] = a.max() - a.min()
    ax.axhline(0, color=GRID, lw=1)
    ax.plot(z, a, color=BLUE, lw=2, marker="o", ms=3)
    for q in np.quantile(X_test[feature], np.linspace(0.1, 0.9, 9)):
        ax.axvline(q, ymin=0, ymax=0.04, color=MUTED, lw=1)
    ax.set_xlim(X_test[feature].quantile(0.01), X_test[feature].quantile(0.99))
    ax.set_xlabel(feature)
    ax.set_ylabel("ALE, сотни тысяч долларов")
fig.suptitle("ALE для California Housing, градиентный бустинг, K = 20; засечки внизу — децили признака, ось x — от 1-го до 99-го процентиля")
fig.tight_layout()
plt.show()
pd.Series(span, name="размах ALE").round(3)

## 5. PDP, M-plot и ALE на скоррелированных признаках

Возьмем модель, которую не нужно обучать: $f(x_1, x_2) = x_1 + x_2 + 5(x_1 - x_2)^2$. Признаки почти совпадают: $x_1 \sim U(-1, 1)$, $x_2 = x_1 + \varepsilon$, $\varepsilon \sim N(0, 0.05^2)$. В области данных $x_1 \approx x_2$, третье слагаемое почти нулевое, и собственный эффект $x_1$ — прямая с наклоном 1. Вне области данных, где $x_1$ и $x_2$ далеки, третье слагаемое большое.

**Задание 5.** Посчитайте по $x_1$ на сетке из 10 точек от $-0.9$ до $0.9$ три кривые: PDP, M-plot (средний прогноз объектов с $|x_1 - v| < 0.1$) и ALE с $K = 10$. Подгоните к каждой кривой параболу и сравните ее коэффициенты с истинными: 0 при $x_1^2$ и 1 при $x_1$.

In [ ]:
rng = np.random.RandomState(0)
x1 = rng.uniform(-1, 1, 2000)
D = pd.DataFrame({"x1": x1, "x2": x1 + rng.normal(0, 0.05, 2000)})


class KnownModel:
    def predict(self, X):
        return X["x1"].values + X["x2"].values + 5 * (X["x1"].values - X["x2"].values) ** 2


known = KnownModel()
v = np.linspace(-0.9, 0.9, 10)
pdp = ...    # ваш код: средний прогноз по всем строкам D, где x1 заменен на t
mplot = ...  # ваш код: средний прогноз строк D, у которых |x1 - t| < 0.1
z, a = ale(known, D, "x1", K=10)

for name, xs, curve in [("PDP", v, pdp), ("M-plot", v, mplot), ("ALE", z, a)]:
    c2, c1, _ = np.polyfit(xs, curve, 2)
    print(f"{name}: коэффициент при x1^2 {c2:.3f}, при x1 {c1:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(v, pdp - pdp.mean(), color=ORANGE, lw=2, label="PDP")
ax.plot(v, mplot - mplot.mean(), color=AQUA, lw=2, label="M-plot")
ax.plot(z, a, color=BLUE, lw=2, label="ALE")
ax.plot(v, v, color=INK, lw=1, ls="--", label="истинный эффект x1")
ax.set_xlabel("x1")
ax.set_ylabel("эффект, центрированный")
ax.set_title("f = x1 + x2 + 5(x1 - x2)^2, x2 = x1 + noise")
ax.legend()
plt.show()

**Место для ваших выводов:** какая из трех кривых восстанавливает собственный эффект $x_1$? Что именно ломает каждую из двух других?

## 6. Выбор числа интервалов $K$

Локальный эффект — среднее по $|N(k)|$ объектам: чем больше $K$, тем меньше объектов в интервале и тем шумнее кривая; чем меньше $K$, тем грубее кривая.

**Задание 6.** Постройте ALE бустинга по `AveOccup` на 1000 тестовых строках при $K = 5$, $20$ и $200$. Для каждого $K$ посчитайте среднее число объектов в интервале и полную вариацию кривой — сумму модулей приращений $\sum_k |A_k - A_{k-1}|$.

In [ ]:
X_k = X_test.sample(1000, random_state=0)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
rows = {}
for K, color in [(5, ORANGE), (20, BLUE), (200, AQUA)]:
    z, a = ale(model, X_k, "AveOccup", K=K)
    rows[K] = ...  # ваш код: среднее число объектов в интервале и полная вариация
    ax.plot(z, a, color=color, lw=1.5, label=f"K = {K}")
ax.set_xlim(X_k["AveOccup"].quantile(0.01), X_k["AveOccup"].quantile(0.99))
ax.set_xlabel("AveOccup")
ax.set_ylabel("ALE, сотни тысяч долларов")
ax.set_title("ALE по AveOccup при разном K, 1000 тестовых строк")
ax.legend()
plt.show()
pd.DataFrame(rows).T.round(3)

**Место для ваших выводов:** как меняется кривая с ростом $K$ и какой $K$ вы бы выбрали для этих 1000 строк?